# Lecture 9: Final Project Experimental Best Practices for Geometric Deep Learning

**Course context.** In geometric deep learning, we do not add symmetry priors (equivariance/invariance) as decoration. We add them to encode a scientific assumption about how the world and the task transform.

A strong final project is therefore **not** defined by a single best metric on a leaderboard. It is defined by a **clear claim**, a **fair test**, and **evidence that matches the claim**.

> **Key question:** *What benefit is the symmetry prior supposed to provide, and under what regime should that benefit appear?*

Throughout this notebook, keep one principle in mind:

- A clean, well-controlled experiment is better than a large, messy benchmark.
- Equivariant models are not guaranteed to win everywhere; they should win in regimes where the inductive bias matches the data-generating structure.

## 1) What makes an experimental claim credible?

A credible claim has four parts:

1. **Hypothesis**: a concrete statement connecting symmetry to expected behavior.
   - Example: "SE(3)-equivariance should improve sample efficiency for molecular force prediction under random rigid transforms."
2. **Controlled comparison**: architecture differences are isolated from unrelated advantages.
3. **Targeted regime**: evaluate where the claimed benefit should appear (OOD rotations, low-data, reduced augmentation, etc.).
4. **Uncertainty-aware reporting**: multiple seeds, variance, and honest effect sizes.

In this course, we care about the chain:

\[
\text{Symmetry assumption} \rightarrow \text{architectural constraint} \rightarrow \text{learning behavior} \rightarrow \text{measured outcome}.
\]

If your results skip a link in this chain, your scientific story is incomplete.

## 2) Fair baselines: compare inductive bias, not hidden advantages

A weak baseline cannot validate an equivariant method. Your baseline should be competitive and honestly tuned.

### Match the following whenever possible

- **Parameter count** (or report clearly when exact matching is impossible).
- **Optimizer** (e.g., AdamW for both models).
- **Learning-rate schedule** (same schedule family and tuning budget).
- **Batch size**.
- **Number of epochs / update steps**.
- **Early stopping policy**.
- **Data augmentation policy**.
- **Accessible input information** (no privileged features for one model only).

### Why this matters

If an equivariant model wins only because it has more parameters, stronger tuning, longer training, or extra information, you have not shown a symmetry benefit. You have shown an unfair setup.

### Practical reporting tip

Include a compact comparison table in your report:

| Model | Params | Optimizer | LR schedule | Epochs | Augmentations | Extra inputs |
|---|---:|---|---|---:|---|---|
| Baseline CNN | ... | ... | ... | ... | ... | ... |
| Rotation-equivariant CNN | ... | ... | ... | ... | ... | ... |

In [ ]:
# Lightweight utility: count trainable parameters (PyTorch)
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Example usage:
# print(f"Trainable params: {count_trainable_params(model):,}")

## 3) Proper train / validation / test separation

### Roles of each split

- **Train**: fit model parameters.
- **Validation**: model selection (hyperparameters, architecture variants, stopping).
- **Test**: one-shot final estimate after decisions are fixed.

### Critical rule

Do **not** use test performance to choose hyperparameters, architecture depth, or augmentation strength. That is test leakage.

### Leakage risks in geometric tasks

- Near-duplicate structures across train/test (e.g., conformers, meshes, scenes).
- Shared objects with transformed copies spread across splits.
- Temporal/spatial correlation that makes random splitting optimistic.

### Why random IID splits can hide equivariance benefits

If train and test contain similar transformations, a non-equivariant model can memorize augmentation patterns and appear strong. The real value of equivariance often appears when transformation coverage at test time differs from train time.

Design splits that test the claim:

- Rotation-range split (train on limited angles, test on unseen angles).
- Group-element holdout (train on subset of transformations, test on held-out subset).
- Size/composition split (train on smaller graphs/molecules, test on larger/novel ones).

## 4) Experiments that expose the value of equivariant architectures

Evaluate at least one regime where symmetry-aware inductive bias should plausibly help:

1. **OOD generalization**
   - Example: unseen rotations, unseen graph sizes, unseen poses.
2. **Sample efficiency**
   - Compare performance vs. training-set size.
3. **Training/optimization efficiency**
   - Compare convergence speed at equal compute budget.
4. **Reduced augmentation dependence**
   - Show whether equivariance reduces heavy augmentation needs.
5. **Equivariance consistency checks**


In [ ]:
# Pseudocode: sample-efficiency sweep + seed aggregation
import numpy as np

train_fracs = [0.1, 0.25, 0.5, 1.0]
seeds = [0, 1, 2, 3, 4]
results = {"baseline": {}, "equivariant": {}}

for frac in train_fracs:
    for model_name in ["baseline", "equivariant"]:
        metrics = []
        for seed in seeds:
            # set_seed(seed)
            # train_loader = make_loader(train_data, frac=frac)
            # model = build_model(model_name)
            # train(model, train_loader, val_loader)
            # metrics.append(evaluate(model, test_loader)["main_metric"])
            pass
        # Replace placeholder with real metric list
        # results[model_name][frac] = (np.mean(metrics), np.std(metrics))

# Report mean ± std at each fraction for both models.

In [ ]:
# Pseudocode: equivariance consistency check
# Goal: small error means model output transforms as expected.

def equivariance_error(model, x, group_action_input, group_action_output, g):
    y1 = model(group_action_input(g, x))
    y2 = group_action_output(g, model(x))
    return (y1 - y2).norm() / (y2.norm() + 1e-8)

# Evaluate over random x and random group elements g, then report mean ± std.

## 5) Worked mini-case studies (course-relevant)

Below are concrete templates you can adapt. Each includes the task, symmetry, fair comparison, and a regime where equivariance should help.

### Example A — Rotation-equivariant image prediction

**Task.** Classify or regress properties of objects where absolute orientation is nuisance (or output rotates predictably).

**Symmetry.** Planar rotations (e.g., \(C_N\) discrete rotations or approximate \(SO(2)\)).

**Architecture comparison.**
- Baseline: standard CNN.
- Symmetry-aware: rotation-equivariant CNN (e.g., group convolutions / steerable filters).

**Fair design.**
- Match trainable parameter count as closely as possible.
- Same optimizer, scheduler, epochs, batch size, and early stopping.
- Same access to inputs and same augmentation policy in each condition.

**Revealing regime (better than plain IID).**
- **OOD rotation split:** train on restricted angles (e.g., \([-30^\circ, 30^\circ]\)), test on unseen angles (e.g., \([60^\circ, 180^\circ]\)).
- Optionally compare two settings:
  1. minimal augmentation,
  2. heavy rotation augmentation.

**Expected interpretation.**
If equivariance helps, gains should be strongest under unseen rotations and/or low augmentation. On IID random splits with full augmentation, the gap may shrink.

### Example B — Permutation-equivariant graph prediction

**Task.** Graph-level property prediction (e.g., synthetic graph classification/regression or molecular graph property prediction with fixed connectivity features).

**Symmetry.** Permutations of node indices. Relabeling nodes should not change graph-level predictions.

**Why symmetry is structurally required.**
Node ordering is arbitrary metadata, not signal. If a model is sensitive to ordering, it can learn spurious patterns tied to indexing conventions.

**Architecture comparison.**
- Symmetry-aware: message passing GNN / Deep Sets-style pooling (permutation equivariant/invariant by construction).
- Weak baseline: MLP on flattened adjacency/features using a fixed node order (order-sensitive).

**Fair design.**
- Comparable parameter counts and training budget.
- Same node/edge features and preprocessing.
- Report performance with and without random index permutations at test time.

**Revealing regime options.**
1. **Robustness to arbitrary indexing:** randomly permute node indices at test time; equivariant model should be stable.
2. **Sample efficiency:** compare performance vs. training size.
3. **Generalization across graph sizes:** train on smaller graphs, test on larger graphs where combinatorial structure changes.

**Expected interpretation.**
Permutation-aware architectures should offer more stable and data-efficient behavior when ordering artifacts are removed.

### Example C — 3D geometric prediction under rigid motions

**Task.** 3D prediction such as molecular energy/force modeling, point cloud property prediction, or rigid-body state estimation.

**Symmetry.** Rigid motions (typically \(SE(3)\): rotations + translations; sometimes reflections depending on physics/task).

**Output behavior.**
- Scalar quantities like energy: often **invariant** under global rigid transforms.
- Vector quantities like forces: should be **equivariant** (rotate/translate consistently).

**Architecture comparison.**
- Baseline: non-equivariant 3D network with coordinate inputs and heavy augmentation.
- Symmetry-aware: \(SE(3)\)-equivariant architecture (e.g., tensor field/irreps-based model, equivariant GNN).

**Fair design.**
- Match parameter count and optimization setup.
- If baseline uses augmentation, document augmentation distribution explicitly.
- Align compute budget (wall-clock or update steps) and report both accuracy and runtime.

**Nontrivial experiment beyond IID.**
- **Transform stress test:** evaluate under random rigid motions not seen during training distribution.
- **Low-data regime:** train with reduced samples and compare error curves.
- **Consistency metric:** measure equivariance error for vector outputs under random rotations.

**Expected interpretation.**
Equivariance should primarily improve transform consistency, robustness, and sample efficiency—not necessarily dominate every IID benchmark metric.

## 6) Ablations and reporting standards

Include ablations that isolate *why* a method works:

- Remove or weaken equivariant components.
- Vary augmentation strength to test whether learned invariance substitutes for built-in equivariance.
- Vary model width/depth under fixed budget.
- Evaluate with multiple random seeds when possible.

Report at minimum:

- Mean ± std over seeds.
- Parameter counts.
- Training budget (epochs/steps, batch size, optimizer/scheduler).
- Runtime or compute discussion when relevant.
- Metrics aligned with claim (e.g., calibration, OOD robustness, equivariance error—not only top-1 accuracy).

### Avoid overclaiming

A tiny improvement (e.g., +0.2%) without uncertainty/context is not strong evidence. Claims should match effect size, variance, and experimental controls.

## 7) Final project checklist (pre-submission)

Use this as a quick audit before submission:

- [ ] I state a **precise hypothesis** linking symmetry prior to expected benefit.
- [ ] I compare against **fair, competitive baselines**.
- [ ] I report or approximately match **parameter counts**.
- [ ] I align **training budget** (optimizer, scheduler, epochs/steps, batch size, early stopping).
- [ ] I maintain a **clean train/val/test split** with no test-driven tuning.
- [ ] I include at least one **symmetry-revealing experiment** (OOD, low-data, reduced augmentation, or consistency test).
- [ ] I include at least one **ablation** isolating the role of equivariance/invariance.
- [ ] I report **metrics aligned with the claim** and include uncertainty (e.g., mean ± std).
- [ ] My conclusions are **proportional to the evidence** and do not overclaim.

---

**Final reminder.** In geometric deep learning, the most convincing project is not the one with the most plots—it is the one where architecture, symmetry assumptions, and empirical evidence form a coherent scientific argument.